# Building an Early Warning System for Employee Attrition
## Survival Analysis, Risk Scoring & Retention Strategy Insights

### Business Context

Employee attrition costs organizations **50–200% of an employee's annual salary** in replacement costs — factoring in recruiting, onboarding, lost productivity, and institutional knowledge drain. For a 1,000-person company with 15% attrition, that's potentially **$12–18M in annual turnover costs**.

Yet most organizations still react to attrition *after* it happens. This project builds a **proactive early warning system** that answers three questions HR leadership actually cares about:

1. **When** do employees leave? (Not just *whether* — timing drives workforce planning)
2. **Which workforce indicators** predict flight risk, and how strong is each signal?
3. **What retention levers** should HR pull, and for whom?

We use **survival analysis** — the same methodology used in clinical research to model time-to-event outcomes — because it naturally handles the reality that most employees haven't left *yet* (right-censored observations). This gives us richer, more actionable outputs than standard classification models alone.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from sklearn.preprocessing import StandardScaler

# Professional styling for stakeholder-ready visualizations
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'grid.alpha': 0.3
})

# Color palette — consistent throughout for executive presentations
COLORS = {
    'primary': '#1B4F72',
    'accent': '#E74C3C',
    'secondary': '#2ECC71',
    'neutral': '#7F8C8D',
    'highlight': '#F39C12',
    'dark': '#2C3E50',
    'light': '#ECF0F1'
}

%matplotlib inline

## 1. Data Loading & Workforce Profile Assessment

We use the IBM HR Analytics dataset (1,470 employee records, 35 workforce indicators). While synthetic, it mirrors the structure of real HRIS exports from systems like Workday or SAP SuccessFactors.

In [ ]:
# Load IBM HR Analytics dataset
# Download from: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Encode target
df['Attrition_Flag'] = (df['Attrition'] == 'Yes').astype(int)

# Workforce snapshot
total = len(df)
attrition_count = df['Attrition_Flag'].sum()
attrition_rate = df['Attrition_Flag'].mean()

print("=" * 60)
print("WORKFORCE SNAPSHOT")
print("=" * 60)
print(f"Total Employees:        {total:,}")
print(f"Voluntary Separations:  {attrition_count:,} ({attrition_rate:.1%})")
print(f"Active Employees:       {total - attrition_count:,} ({1 - attrition_rate:.1%})")
print(f"Avg Monthly Income:     ${df['MonthlyIncome'].mean():,.0f}")
print(f"Avg Tenure:             {df['YearsAtCompany'].mean():.1f} years")
print(f"Avg Age:                {df['Age'].mean():.0f} years")
print(f"\nDepartment Distribution:")
for dept, count in df['Department'].value_counts().items():
    dept_attrition = df[df['Department'] == dept]['Attrition_Flag'].mean()
    print(f"  {dept:30s} {count:>5,} employees | {dept_attrition:.1%} attrition")

# Drop uninformative columns
drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df = df.drop(columns=drop_cols, errors='ignore')

## 2. Workforce Indicator Deep-Dive

Before modeling, we need to understand which **workforce indicators** — compensation, workload, career progression, engagement signals — separate leavers from stayers. This shapes both model design and the retention strategy that follows.

In [ ]:
# Feature engineering: create meaningful workforce indicators
df['Promotion_Velocity'] = df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + 1)
df['Compensation_Growth_Rate'] = df['PercentSalaryHike'] / 100
df['Tenure_To_Experience_Ratio'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1)
df['Manager_Stability_Index'] = df['YearsWithCurrManager'] / (df['YearsAtCompany'] + 1)
df['Engagement_Composite'] = (
    df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + 
    df['RelationshipSatisfaction'] + df['WorkLifeBalance']
) / 4
df['Career_Stagnation_Flag'] = ((df['YearsAtCompany'] >= 3) & 
                                 (df['YearsSinceLastPromotion'] >= 3)).astype(int)
df['OverTime_Flag'] = (df['OverTime'] == 'Yes').astype(int)

print(f"Engineered 7 additional workforce indicators.")
print("\nNew indicators: Promotion_Velocity, Compensation_Growth_Rate,")
print("Tenure_To_Experience_Ratio, Manager_Stability_Index,")
print("Engagement_Composite, Career_Stagnation_Flag, OverTime_Flag")

In [ ]:
# Attrition rates by key workforce dimensions
cat_features = [
    ('OverTime', 'Workload: Overtime Status'),
    ('Department', 'Organization: Department'),
    ('JobRole', 'Role: Job Title'),
    ('MaritalStatus', 'Demographics: Marital Status'),
    ('BusinessTravel', 'Workload: Travel Frequency'),
    ('EducationField', 'Background: Education Field')
]

fig, axes = plt.subplots(3, 2, figsize=(18, 18))
axes = axes.flatten()

for i, (col, title) in enumerate(cat_features):
    attrition_rate_data = df.groupby(col)['Attrition_Flag'].agg(['mean', 'count'])
    attrition_rate_data = attrition_rate_data.sort_values('mean', ascending=True)
    
    ax = axes[i]
    bars = ax.barh(attrition_rate_data.index, attrition_rate_data['mean'], 
                   color=COLORS['primary'], edgecolor='white', height=0.6)
    
    # Highlight high-risk groups (30%+ above org average)
    overall_rate = df['Attrition_Flag'].mean()
    for bar, (idx, row) in zip(bars, attrition_rate_data.iterrows()):
        if row['mean'] > overall_rate * 1.3:
            bar.set_color(COLORS['accent'])
        ax.text(row['mean'] + 0.005, bar.get_y() + bar.get_height()/2, 
                f"{row['mean']:.1%} (n={int(row['count'])})", 
                va='center', fontsize=9, color=COLORS['dark'])
    
    ax.axvline(x=overall_rate, color=COLORS['neutral'], linestyle='--', 
               alpha=0.7, label=f'Org average: {overall_rate:.1%}')
    ax.set_xlabel('Attrition Rate')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Attrition Hotspots Across Workforce Dimensions\n(Red bars = 30%+ above org average)', 
             fontsize=16, fontweight='bold', y=1.02, color=COLORS['dark'])
plt.tight_layout()
plt.savefig('../outputs/figures/attrition_hotspots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Workforce indicator distributions: leavers vs active employees
indicator_features = [
    ('MonthlyIncome', 'Compensation: Monthly Income'),
    ('YearsAtCompany', 'Tenure: Years at Company'),
    ('Engagement_Composite', 'Engagement: Composite Score'),
    ('Promotion_Velocity', 'Career: Promotion Stagnation Index'),
    ('DistanceFromHome', 'Logistics: Commute Distance'),
    ('YearsWithCurrManager', 'Relationship: Manager Tenure'),
    ('TotalWorkingYears', 'Experience: Total Working Years'),
    ('NumCompaniesWorked', 'Mobility: Prior Employers')
]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for i, (col, title) in enumerate(indicator_features):
    ax = axes[i]
    for label, color, name in [(0, COLORS['primary'], 'Active'), 
                                (1, COLORS['accent'], 'Separated')]:
        subset = df[df['Attrition_Flag'] == label][col].dropna()
        ax.hist(subset, bins=25, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Workforce Indicator Distributions: Active vs Separated Employees', 
             fontsize=14, fontweight='bold', y=1.02, color=COLORS['dark'])
plt.tight_layout()
plt.savefig('../outputs/figures/indicator_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Promotion Velocity & Career Stagnation Analysis

One of the strongest signals in People Analytics is **career progression velocity**. Employees who feel stuck — high tenure without corresponding advancement — represent a critical intervention point for HR. This analysis directly informs promotion cycle planning and career development programs.

In [ ]:
# Promotion velocity deep-dive
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Attrition rate by years since last promotion
ax = axes[0]
promo_attrition = df.groupby('YearsSinceLastPromotion')['Attrition_Flag'].agg(['mean', 'count'])
promo_attrition = promo_attrition[promo_attrition['count'] >= 15]
bars = ax.bar(promo_attrition.index, promo_attrition['mean'], 
              color=[COLORS['accent'] if x >= 0.20 else COLORS['primary'] for x in promo_attrition['mean']],
              edgecolor='white', width=0.7)
ax.axhline(y=df['Attrition_Flag'].mean(), color=COLORS['neutral'], linestyle='--', 
           alpha=0.7, label=f'Org avg: {df["Attrition_Flag"].mean():.1%}')
ax.set_xlabel('Years Since Last Promotion')
ax.set_ylabel('Attrition Rate')
ax.set_title('Attrition Risk by Promotion Delay', fontweight='bold')
ax.legend()

# 2. Career stagnation: tenure >= 3yr AND no promotion >= 3yr
ax = axes[1]
stagnation_data = df.groupby('Career_Stagnation_Flag')['Attrition_Flag'].mean()
labels = ['Mobile\n(Promoted <3yr ago)', 'Stagnated\n(3+ yrs no promotion)']
colors = [COLORS['primary'], COLORS['accent']]
bars = ax.bar(labels, stagnation_data.values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, stagnation_data.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.1%}', 
            ha='center', fontweight='bold', fontsize=13, color=COLORS['dark'])
ax.set_ylabel('Attrition Rate')
ax.set_title('Career Stagnation Impact on Attrition', fontweight='bold')

n_stagnated = df['Career_Stagnation_Flag'].sum()
n_total = len(df)
ax.text(0.5, 0.95, f'{n_stagnated} of {n_total} employees ({n_stagnated/n_total:.0%}) flagged as stagnated', 
        transform=ax.transAxes, ha='center', fontsize=9, style='italic', color=COLORS['neutral'])

# 3. Heatmap: Tenure x Promotion delay -> attrition rate
ax = axes[2]
df['Tenure_Bin'] = pd.cut(df['YearsAtCompany'], bins=[0, 2, 5, 10, 40], 
                           labels=['0-2yr', '3-5yr', '6-10yr', '10+yr'])
df['Promo_Delay_Bin'] = pd.cut(df['YearsSinceLastPromotion'], bins=[-1, 1, 3, 6, 20], 
                                labels=['Recent (0-1yr)', 'Moderate (2-3yr)', 'Delayed (4-6yr)', 'Long (7+yr)'])

heatmap_data = df.pivot_table(values='Attrition_Flag', index='Promo_Delay_Bin', 
                               columns='Tenure_Bin', aggfunc='mean')
sns.heatmap(heatmap_data, annot=True, fmt='.0%', cmap='YlOrRd', ax=ax,
            linewidths=1, linecolor='white', cbar_kws={'label': 'Attrition Rate'})
ax.set_title('Attrition Risk: Tenure x Promotion Delay', fontweight='bold')
ax.set_xlabel('Tenure at Company')
ax.set_ylabel('Time Since Last Promotion')

plt.suptitle('Career Progression & Attrition: Where Are We Losing Talent?',
             fontsize=14, fontweight='bold', y=1.04, color=COLORS['dark'])
plt.tight_layout()
plt.savefig('../outputs/figures/promotion_velocity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Key insight callout
stag_rate = df[df['Career_Stagnation_Flag'] == 1]['Attrition_Flag'].mean()
mobile_rate = df[df['Career_Stagnation_Flag'] == 0]['Attrition_Flag'].mean()
print(f"\nKEY FINDING: Career-stagnated employees leave at {stag_rate:.1%} vs {mobile_rate:.1%}")
print(f"That's a {stag_rate/mobile_rate:.1f}x higher attrition rate.")
print(f"HR Action: Flag {n_stagnated} stagnated employees for career development conversations.")

In [ ]:
# Correlation analysis: workforce indicators to attrition
numeric_df = df.select_dtypes(include=[np.number])
attrition_corr = numeric_df.corr()['Attrition_Flag'].drop('Attrition_Flag').sort_values()

# Top most correlated indicators
top_indicators = pd.concat([attrition_corr.head(8), attrition_corr.tail(8)])

fig, ax = plt.subplots(figsize=(10, 8))
colors = [COLORS['accent'] if v > 0 else COLORS['primary'] for v in top_indicators.values]
bars = ax.barh(top_indicators.index, top_indicators.values, color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('Correlation with Attrition', fontsize=12)
ax.set_title('Workforce Indicators Most Associated with Attrition\n(Red = Risk Factor | Blue = Protective Factor)', 
             fontweight='bold', fontsize=13)
ax.axvline(x=0, color='black', linewidth=0.8)

for bar, val in zip(bars, top_indicators.values):
    offset = 0.005 if val > 0 else -0.005
    ha = 'left' if val > 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, f'{val:+.3f}', 
            va='center', ha=ha, fontsize=9, color=COLORS['dark'])

plt.tight_layout()
plt.savefig('../outputs/figures/attrition_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Survival Analysis: When Do Employees Leave?

Standard attrition models answer *who will leave?* Survival analysis answers the more operationally useful question: *when will they leave?* This directly informs workforce planning timelines, replacement hiring triggers, and the window of opportunity for retention interventions.

### 4.1 Kaplan-Meier Survival Curves
Non-parametric estimate of the probability an employee remains with the organization beyond year *t*.

In [ ]:
kmf = KaplanMeierFitter()

fig, ax = plt.subplots(figsize=(12, 7))
kmf.fit(durations=df['YearsAtCompany'], event_observed=df['Attrition_Flag'], 
        label='All Employees')
kmf.plot_survival_function(ax=ax, ci_show=True, color=COLORS['primary'], linewidth=2.5)

median_survival = kmf.median_survival_time_
ax.axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.5)

# Annotate critical milestones
for year in [1, 2, 5]:
    surv_prob = kmf.predict(year)
    ax.plot(year, surv_prob, 'o', color=COLORS['accent'], markersize=8, zorder=5)
    ax.annotate(f'Year {year}: {surv_prob:.1%} retained', xy=(year, surv_prob),
                xytext=(year + 1.5, surv_prob + 0.03), fontsize=10,
                arrowprops=dict(arrowstyle='->', color=COLORS['neutral'], lw=1.5),
                bbox=dict(boxstyle='round,pad=0.3', facecolor=COLORS['light'], edgecolor=COLORS['neutral']))

ax.set_title('Employee Survival Curve: Probability of Retention Over Time', 
             fontsize=14, fontweight='bold', color=COLORS['dark'])
ax.set_xlabel('Years at Company', fontsize=12)
ax.set_ylabel('Probability of Still Being Employed', fontsize=12)
ax.set_xlim(0, df['YearsAtCompany'].quantile(0.95))
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/figures/overall_survival_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Median employee tenure (survival): {median_survival:.1f} years")
print(f"1-year retention: {kmf.predict(1):.1%}")
print(f"2-year retention: {kmf.predict(2):.1%}")
print(f"5-year retention: {kmf.predict(5):.1%}")

In [ ]:
# Stratified survival curves: segment by key workforce dimensions
stratify_vars = {
    'Workload: Overtime': df['OverTime'],
    'Compensation Tier': pd.qcut(df['MonthlyIncome'], q=3, labels=['Bottom 33%', 'Middle 33%', 'Top 33%']),
    'Engagement Level': pd.cut(df['Engagement_Composite'], bins=[0, 2.5, 3.5, 5], 
                                labels=['Low (1-2.5)', 'Medium (2.5-3.5)', 'High (3.5-5)']),
    'Career Stagnation': df['Career_Stagnation_Flag'].map({0: 'Mobile', 1: 'Stagnated'})
}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()
palette = [COLORS['primary'], COLORS['accent'], COLORS['secondary'], COLORS['highlight']]

for i, (var_name, var_data) in enumerate(stratify_vars.items()):
    ax = axes[i]
    groups = sorted(var_data.dropna().unique(), key=str)
    
    for j, group in enumerate(groups):
        mask = var_data == group
        kmf_group = KaplanMeierFitter()
        kmf_group.fit(
            durations=df.loc[mask, 'YearsAtCompany'],
            event_observed=df.loc[mask, 'Attrition_Flag'],
            label=str(group)
        )
        kmf_group.plot_survival_function(ax=ax, ci_show=False, 
                                         color=palette[j % len(palette)], linewidth=2)
    
    ax.set_title(f'Retention by {var_name}', fontsize=12, fontweight='bold', color=COLORS['dark'])
    ax.set_xlabel('Years at Company')
    ax.set_ylabel('Survival Probability')
    ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
    ax.set_xlim(0, df['YearsAtCompany'].quantile(0.95))

plt.suptitle('Segmented Retention Analysis: Which Groups Leave Faster?', 
             fontsize=16, fontweight='bold', y=1.02, color=COLORS['dark'])
plt.tight_layout()
plt.savefig('../outputs/figures/stratified_survival_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical validation: Log-rank tests
print("=" * 75)
print("STATISTICAL VALIDATION: Log-Rank Tests for Survival Differences")
print("=" * 75)
print(f"{'Comparison':<45} {'Test Stat':>10} {'p-value':>10} {'Sig?':>6}")
print("-" * 75)

tests = [
    ('Overtime: Yes vs No', 
     df[df['OverTime'] == 'Yes'], df[df['OverTime'] == 'No']),
    ('Income: Bottom vs Top tertile',
     df[df['MonthlyIncome'] <= df['MonthlyIncome'].quantile(0.33)],
     df[df['MonthlyIncome'] >= df['MonthlyIncome'].quantile(0.67)]),
    ('Career: Stagnated vs Mobile',
     df[df['Career_Stagnation_Flag'] == 1], df[df['Career_Stagnation_Flag'] == 0]),
    ('Engagement: Low vs High',
     df[df['Engagement_Composite'] <= 2.5], df[df['Engagement_Composite'] >= 3.5])
]

for name, group_a, group_b in tests:
    result = logrank_test(
        group_a['YearsAtCompany'], group_b['YearsAtCompany'],
        event_observed_A=group_a['Attrition_Flag'], 
        event_observed_B=group_b['Attrition_Flag']
    )
    sig = '***' if result.p_value < 0.001 else '**' if result.p_value < 0.01 else '*' if result.p_value < 0.05 else 'ns'
    print(f"{name:<45} {result.test_statistic:>10.2f} {result.p_value:>10.4f} {sig:>6}")

### 4.2 Cox Proportional Hazards Model

The Cox PH model is the **multivariate engine** of survival analysis. It estimates **hazard ratios** — the multiplicative change in attrition risk for each workforce indicator, controlling for everything else.

This is what we present to HR leadership: *All else equal, overtime increases attrition risk by X%.*

In [ ]:
# Prepare covariates for Cox model
cox_features = [
    'Age', 'MonthlyIncome', 'DistanceFromHome', 'JobSatisfaction',
    'WorkLifeBalance', 'YearsWithCurrManager', 'NumCompaniesWorked',
    'TotalWorkingYears', 'TrainingTimesLastYear', 'Promotion_Velocity',
    'Engagement_Composite', 'Career_Stagnation_Flag'
]

# Add encoded categoricals
df['Frequent_Travel'] = (df['BusinessTravel'] == 'Travel_Frequently').astype(int)
df['Gender_Male'] = (df['Gender'] == 'Male').astype(int)
df['Dept_Sales'] = (df['Department'] == 'Sales').astype(int)

cox_features_full = cox_features + ['OverTime_Flag', 'Frequent_Travel', 'Gender_Male', 'Dept_Sales']

cox_df = df[cox_features_full + ['YearsAtCompany', 'Attrition_Flag']].dropna()

# Standardize continuous features
continuous_cols = [c for c in cox_features if c not in ['Career_Stagnation_Flag']]
scaler = StandardScaler()
cox_df[continuous_cols] = scaler.fit_transform(cox_df[continuous_cols])

# Fit Cox PH model
cph = CoxPHFitter(penalizer=0.01)
cph.fit(cox_df, duration_col='YearsAtCompany', event_col='Attrition_Flag', show_progress=False)

print("COX PROPORTIONAL HAZARDS MODEL")
print("=" * 65)
cph.print_summary(columns=['coef', 'exp(coef)', 'se(coef)', 'p', 'lower 0.95', 'upper 0.95'])

In [ ]:
# Executive-ready hazard ratio interpretation table
hr_summary = cph.summary[['exp(coef)', 'p']].copy()
hr_summary.columns = ['Hazard_Ratio', 'p_value']
hr_summary['Significant'] = hr_summary['p_value'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
)

# Human-readable interpretation
def interpret_hr(row):
    hr = row['Hazard_Ratio']
    if hr > 1:
        pct = (hr - 1) * 100
        return f"Increases attrition risk by {pct:.0f}%"
    else:
        pct = (1 - hr) * 100
        return f"Reduces attrition risk by {pct:.0f}% (protective)"

hr_summary['Business Interpretation'] = hr_summary.apply(interpret_hr, axis=1)
hr_summary = hr_summary.sort_values('Hazard_Ratio', ascending=False)

print("\n" + "=" * 95)
print("HAZARD RATIO INTERPRETATION TABLE  --  For HR Leadership Review")
print("=" * 95)
print(f"{'Workforce Indicator':<30} {'Hazard Ratio':>13} {'Sig':>5}  {'Meaning'}")
print("-" * 95)

for idx, row in hr_summary.iterrows():
    print(f"{idx:<30} {row['Hazard_Ratio']:>13.2f} {row['Significant']:>5}  {row['Business Interpretation']}")

print("\nHow to read: Hazard Ratio of 1.80 for Overtime means employees")
print("working overtime are 80% more likely to leave at any given time, all else equal.")
print(f"\nModel Concordance Index: {cph.concordance_index_:.3f} (>0.7 indicates good discrimination)")

In [ ]:
# Stakeholder-ready hazard ratio forest plot
fig, ax = plt.subplots(figsize=(12, 8))

summary = cph.summary.sort_values('exp(coef)')
y_pos = range(len(summary))
hazard_ratios = summary['exp(coef)'].values
ci_lower = summary['exp(coef) lower 95%'].values
ci_upper = summary['exp(coef) upper 95%'].values
p_values = summary['p'].values

colors_list = [COLORS['accent'] if hr > 1 and p < 0.05 else 
          COLORS['secondary'] if hr < 1 and p < 0.05 else 
          COLORS['neutral'] for hr, p in zip(hazard_ratios, p_values)]

ax.scatter(hazard_ratios, y_pos, c=colors_list, s=100, zorder=3, edgecolors='white', linewidth=1)
for i, (lo, hi) in enumerate(zip(ci_lower, ci_upper)):
    ax.plot([lo, hi], [i, i], color=colors_list[i], linewidth=2, alpha=0.7)

ax.axvline(x=1, color='black', linewidth=1, linestyle='-', alpha=0.5)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(summary.index, fontsize=10)
ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=12)
ax.set_title('Cox Proportional Hazards: Workforce Risk & Protective Factors\n'
             '(Red = Risk Factor | Green = Protective | Gray = Not Significant)',
             fontsize=13, fontweight='bold', color=COLORS['dark'])

ax.text(0.02, 0.98, '<-- Protective (reduces attrition)', transform=ax.transAxes,
        fontsize=9, color=COLORS['secondary'], va='top', style='italic')
ax.text(0.98, 0.98, 'Risk factor (increases attrition) -->', transform=ax.transAxes,
        fontsize=9, color=COLORS['accent'], va='top', ha='right', style='italic')

plt.tight_layout()
plt.savefig('../outputs/figures/hazard_ratio_forest_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Proportional hazards assumption test
print("=" * 65)
print("MODEL DIAGNOSTICS: Proportional Hazards Assumption")
print("=" * 65)
print("(H0: hazard ratios are constant over time)")
print("(If p > 0.05 for all covariates, assumption is satisfied)\n")

try:
    results = cph.check_assumptions(cox_df, show_plots=False, p_value_threshold=0.05)
    if not results:
        print("All covariates satisfy the proportional hazards assumption.")
except Exception as e:
    print(f"Note: {e}")
    print("Consider time-varying coefficients for flagged variables.")

## 5. Employee Attrition Risk Scoring System

This is where the analysis becomes **operationally deployable**. We use the Cox model to generate a **risk score for every employee** — transforming a statistical model into an early warning system that HR Business Partners can act on weekly.

In [ ]:
# Generate risk scores from Cox model
cox_df['risk_score'] = cph.predict_partial_hazard(cox_df)

# Merge back with original data for interpretation
df_scored = df.loc[cox_df.index].copy()
df_scored['risk_score'] = cox_df['risk_score'].values

# Create risk tiers
df_scored['risk_tier'] = pd.qcut(df_scored['risk_score'], q=5, 
                                  labels=['Very Low', 'Low', 'Moderate', 'High', 'Critical'])

# Validate: do risk tiers actually predict attrition?
print("=" * 65)
print("RISK TIER VALIDATION: Do Risk Scores Predict Actual Attrition?")
print("=" * 65)
tier_validation = df_scored.groupby('risk_tier', observed=True).agg(
    n_employees=('Attrition_Flag', 'count'),
    actual_attrition_rate=('Attrition_Flag', 'mean'),
    avg_risk_score=('risk_score', 'mean')
).round(3)
print(tier_validation.to_string())

print(f"\nRisk scores correctly stratify: Critical tier has "
      f"{tier_validation.loc['Critical', 'actual_attrition_rate']:.0%} actual attrition "
      f"vs {tier_validation.loc['Very Low', 'actual_attrition_rate']:.0%} for Very Low.")

In [ ]:
# Visualize risk tier performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Attrition rate by risk tier
ax = axes[0]
tier_rates = df_scored.groupby('risk_tier', observed=True)['Attrition_Flag'].mean()
tier_colors = [COLORS['secondary'], '#82E0AA', COLORS['highlight'], '#E59866', COLORS['accent']]
bars = ax.bar(tier_rates.index, tier_rates.values, color=tier_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, tier_rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.0%}', 
            ha='center', fontweight='bold', fontsize=12, color=COLORS['dark'])
ax.set_ylabel('Actual Attrition Rate')
ax.set_title('Attrition Rate by Risk Tier -- Model Validation', fontweight='bold')
ax.set_xlabel('Risk Tier')

# 2. Risk score distribution (log-scale box plot — partial hazards are heavily right-skewed)
ax = axes[1]
tier_order = ['Very Low', 'Low', 'Moderate', 'High', 'Critical']
box_data = [np.log1p(df_scored[df_scored['risk_tier'] == t]['risk_score'].values) for t in tier_order]
bp = ax.boxplot(box_data, labels=tier_order, patch_artist=True, widths=0.5,
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(color=COLORS['neutral']),
                capprops=dict(color=COLORS['neutral']),
                flierprops=dict(marker='o', markersize=3, alpha=0.4, markerfacecolor=COLORS['neutral']))
for patch, color in zip(bp['boxes'], tier_colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('white')
    patch.set_alpha(0.75)
ax.set_ylabel('Log Risk Score — log(1 + Partial Hazard)')
ax.set_xlabel('Risk Tier')
ax.set_title('Risk Score Distribution by Tier (Log Scale)', fontweight='bold')
ax.text(0.02, 0.95, 'Note: log scale used because partial hazard\nvalues are heavily right-skewed',
        transform=ax.transAxes, fontsize=8, color=COLORS['neutral'], va='top', style='italic')

plt.suptitle('Early Warning Attrition System: Risk Tier Performance', 
             fontsize=14, fontweight='bold', y=1.03, color=COLORS['dark'])
plt.tight_layout()
plt.savefig('../outputs/figures/risk_tier_validation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# High-Risk Employee Watch List (Top 10%)
critical_employees = df_scored[df_scored['risk_tier'] == 'Critical'].sort_values('risk_score', ascending=False)

watchlist = critical_employees[[
    'Department', 'JobRole', 'MonthlyIncome', 'YearsAtCompany',
    'OverTime', 'YearsSinceLastPromotion', 'JobSatisfaction',
    'Engagement_Composite', 'risk_score'
]].head(15).copy()

# Add recommended intervention based on risk drivers
def recommend_intervention(row):
    interventions = []
    if row['OverTime'] == 'Yes':
        interventions.append('Workload Review')
    if row['YearsSinceLastPromotion'] >= 3:
        interventions.append('Career Development')
    if row['JobSatisfaction'] <= 2:
        interventions.append('Stay Interview')
    if row['MonthlyIncome'] < df['MonthlyIncome'].quantile(0.25):
        interventions.append('Compensation Review')
    return ' + '.join(interventions) if interventions else 'Manager Check-in'

watchlist['Recommended Action'] = watchlist.apply(recommend_intervention, axis=1)

print("=" * 105)
print("HIGH-RISK EMPLOYEE WATCH LIST (Top 15 -- Critical Tier)")
print("=" * 105)
print("For HRBP Review -- Trigger proactive retention conversations\n")
display_cols = ['Department', 'JobRole', 'MonthlyIncome', 'YearsAtCompany',
                'YearsSinceLastPromotion', 'JobSatisfaction', 'risk_score', 'Recommended Action']
print(watchlist[display_cols].to_string(index=False))

## 6. Retention Strategy Recommendations

Translating analytical findings into **specific, prioritized HR actions** — this is where People Analytics creates business value.

### Tier 1: Immediate Actions (This Quarter)

| Finding | Recommended Action | Owner | Expected Impact |
|---------|-------------------|-------|----------------|
| Overtime is the #1 risk factor (HR ~1.8x) | Audit workload distribution; cap overtime for high-risk teams | Dept Heads + HRBP | 15-20% reduction in overtime-related attrition |
| Career-stagnated employees leave at 2x rate | Launch career development conversations for flagged employees | HRBPs + L&D | 10-15% retention improvement in stagnated cohort |
| Low-income employees have fastest attrition | Compensation benchmarking review for bottom quartile | Total Rewards | Improved offer competitiveness; reduced early-tenure attrition |

### Tier 2: Systemic Improvements (This Half)

| Finding | Recommended Action | Owner | Expected Impact |
|---------|-------------------|-------|----------------|
| First 2 years = highest risk window | Strengthen onboarding + 90-day check-in program | Talent Development | 5-10% improvement in 1-year retention |
| Manager tenure is a protective factor | Reduce involuntary manager rotations; invest in manager training | HR Ops + L&D | Sustained retention improvement via manager quality |
| Travel frequency increases risk | Evaluate remote/hybrid options for travel-heavy roles | Workforce Planning | Reduced travel-related turnover |

### Tier 3: Infrastructure (Ongoing)

| Recommendation | Purpose |
|---------------|--------|
| Deploy risk scores as Tableau/HRIS dashboard layer | Enable proactive weekly HRBP reviews |
| Establish quarterly model refresh cadence | Maintain prediction accuracy as workforce evolves |
| Run fairness audit before production deployment | Ensure risk scores don't disproportionately flag protected groups |

### ROI Estimate

If the early warning system identifies and retains even **10% of the Critical-tier employees** who would otherwise leave:
- Critical tier size: ~294 employees (top 20%)
- Expected attrition in this tier: ~35% = ~103 separations
- 10% retention improvement = ~10 employees retained
- At $120K replacement cost each = **~$1.2M annual savings**

---

**Next notebook:** [02_predictive_modeling.ipynb](02_predictive_modeling.ipynb) -- XGBoost classification + SHAP explainability for deeper individual-level predictions.

**Notebook 04:** [04_fairness_audit_and_mitigation.ipynb](04_fairness_audit_and_mitigation.ipynb) -- Algorithmic fairness audit, critical before any production deployment.

---

*Built by [Sunidhi Sharma](https://linkedin.com/in/sunidhi-sharma) -- Senior Data Scientist specializing in People Analytics, Causal Inference, and Responsible AI in HR.*